# 01. Ingestão de Dados (Call Center Logs)
Neste notebook geramos uma massa de dados sintética de logs de Call Center.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import rand, expr, col, when, date_add, current_date

spark = SparkSession.builder.appName("MLOps Ingestion").getOrCreate()

# Configuração de Ambiente
ENVIRONMENT = "databricks_volume"
BASE_PATH = "/Volumes/workspace/default/raw_data" if ENVIRONMENT == "databricks_volume" else "file:///tmp/mlops"

raw_mlops_path = f"{BASE_PATH}/mlops/raw_call_logs"
print(f"Path alvo: {raw_mlops_path}")

In [ ]:
print("Gerando logs de Call Center...")
num_records = 500000

# Gerando dados base
df = spark.range(0, num_records) \
    .withColumn("call_id", expr("uuid()")) \
    .withColumn("customer_id", (rand() * 50000).cast("int").cast("string")) \
    .withColumn("call_date", date_add(current_date(), -(rand() * 90).cast("int"))) \
    .withColumn("subject_rand", rand()) \
    .withColumn("subject", when(col("subject_rand") < 0.3, "Cobrança")
                          .when(col("subject_rand") < 0.6, "Suporte Técnico")
                          .when(col("subject_rand") < 0.8, "Cancelamento")
                          .otherwise("Informações")) \
    .withColumn("queue_wait_time", (rand() * 600).cast("int")) \
    .withColumn("duration_sec", (rand() * 1800).cast("int")) \
    .withColumn("transfer_count", (rand() * 4).cast("int")) \
    .withColumn("resolution_rand", rand()) \
    .withColumn("resolution_status", when(col("resolution_rand") < 0.7, "Resolvido").otherwise("Pendente"))

# Injeção da Regra de Negócio Crítica (Risco de Bacen)
# Se o assunto for Cobrança/Cancelamento E não foi resolvido E teve alto tempo de espera, o risco sobe!
df = df.withColumn(
    "bacen_complaint",
    when((col("subject").isin("Cobrança", "Cancelamento")) & (col("resolution_status") == "Pendente") & (col("queue_wait_time") > 300),
         when(rand() < 0.8, 1).otherwise(0)
    ).otherwise(
         when(rand() < 0.05, 1).otherwise(0)
    )
).drop("subject_rand", "resolution_rand")

In [ ]:
print("Salvando log bruto (Bronze/Silver)...")
df.write.mode("overwrite").parquet(raw_mlops_path)
print("Sucesso!")
df.show(5)